# Пробуем улучшить baseline

Для этого пробуем применить простую линейную регрессию, сезонную модель и xgboost а так же их комбинации в виде ансамбля моделей и стекинга.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, precision_score, recall_score, f1_score
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from tqdm import tqdm
from scipy.optimize import minimize
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

In [7]:
train_df = pd.read_csv('data/MR_number_train_0w-5w.csv.zip', index_col=0)
test_df = pd.read_csv('data/MR_number_test_5w-6w.csv.zip', index_col=0)

train_data = train_df.values.astype(np.float32)
test_data = test_df.values.astype(np.float32)

VAL_HORIZON = 168
THRESHOLD = 0.5
train_part = train_data[:-VAL_HORIZON]
val_part = train_data[-VAL_HORIZON:]

## Сначала пробуем сезонную модель

In [8]:
class LearnableSeasonal:
    def fit(self, train_arr, val_arr):

        def loss(weights):
            d_w, w_w = weights
            preds = []
            for t in range(len(val_arr)):
                d_idx = len(train_arr) - 24 + (t % 24)
                w_idx = len(train_arr) - 168 + (t % 168)
                d_val = train_arr[d_idx]
                w_val = train_arr[w_idx]
                preds.append(d_w * d_val + w_w * w_val)
            preds = np.array(preds)
            return mean_absolute_error(val_arr.flatten(), preds.flatten())

        res = minimize(loss, [0.5, 0.5], bounds=[(0,1),(0,1)])
        self.dw, self.ww = res.x
        print(f"Seasonal weights: {self.dw:.3f}, {self.ww:.3f}")

    def predict(self, train_arr, steps):
        preds = []
        for t in range(steps):
            d_idx = len(train_arr) - 24 + (t % 24)
            w_idx = len(train_arr) - 168 + (t % 168)
            d_val = train_arr[d_idx]
            w_val = train_arr[w_idx]
            preds.append(self.dw * d_val + self.ww * w_val)
        return np.array(preds)

In [9]:
seasonal = LearnableSeasonal()
seasonal.fit(train_part, val_part)

seasonal_val = seasonal.predict(train_part, VAL_HORIZON)
seasonal_test = seasonal.predict(train_data, len(test_data))

Seasonal weights: 0.444, 0.506


### Считаем метрики для сезонной модели на тестовой выборке

In [10]:
mae = mean_absolute_error(test_data.flatten(), seasonal_test.flatten())
rmse = np.sqrt(np.mean((test_data - seasonal_test)**2))

y_true = (test_data > THRESHOLD).flatten()
y_pred = (seasonal_test > THRESHOLD).flatten()

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"LearnableSeasonal MAE: {mae:.4f}")
print(f"LearnableSeasonal RMSE: {rmse:.4f}")
print(f"LearnableSeasonal Precision: {precision:.4f}")
print(f"LearnableSeasonal Recall: {recall:.4f}")
print(f"LearnableSeasonal F1: {f1:.4f}")

LearnableSeasonal MAE: 0.2405
LearnableSeasonal RMSE: 0.4177
LearnableSeasonal Precision: 0.8288
LearnableSeasonal Recall: 0.8038
LearnableSeasonal F1: 0.8161


## Пробуем линейную регрессию

In [11]:
def prepare_lr_features(data, window=24):
    X, y = [], []

    for t in range(window, len(data)):
        f = data[t-window:t].flatten()
        f = np.append(f, data[t-window:t].mean(axis=0))
        f = np.append(f, data[t-window:t].std(axis=0))

        X.append(f)
        y.append(data[t])

    return np.array(X), np.array(y)

In [12]:
X_lr, y_lr = prepare_lr_features(train_part)
lr = Ridge(alpha=0.1)
lr.fit(X_lr, y_lr)

def rollout_lr(model, history, steps):
    preds = []
    window = history.copy()
    for _ in range(steps):
        f = window.flatten()
        f = np.append(f, window.mean(axis=0))
        f = np.append(f, window.std(axis=0))
        p = model.predict(f.reshape(1,-1))[0]
        preds.append(p)
        window = np.vstack([window[1:], p])
    return np.maximum(np.array(preds), 0)

lr_val = rollout_lr(lr, train_part[-24:], VAL_HORIZON)
lr_test = rollout_lr(lr, train_data[-24:], len(test_data))

### Считаем метрики для линейной регресси на тестовой выборке

In [13]:
mae = mean_absolute_error(test_data.flatten(), lr_test.flatten())
rmse = np.sqrt(np.mean((test_data - lr_test)**2))

y_true = (test_data > THRESHOLD).flatten()
y_pred = (lr_test > THRESHOLD).flatten()

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Ridge MAE: {mae:.4f}")
print(f"Ridge RMSE: {rmse:.4f}")
print(f"Ridge Precision: {precision:.4f}")
print(f"Ridge Recall: {recall:.4f}")
print(f"Ridge F1: {f1:.4f}")

Ridge MAE: 0.2381
Ridge RMSE: 0.4263
Ridge Precision: 0.8177
Ridge Recall: 0.8344
Ridge F1: 0.8260


## XGBoost

In [14]:
def create_features_vectorized(beam_log, start_idx=168):
    n = len(beam_log) - start_idx
    X = np.zeros((n, 171))
    for t in range(start_idx, len(beam_log)):
        i = t - start_idx
        X[i, :168] = beam_log[t-168:t]
        X[i, 168] = beam_log[t-24:t].mean()
        X[i, 169] = beam_log[t-168:t].mean()
        X[i, 170] = beam_log[t-24:t].std()
    return X

In [15]:
def run_xgb(data, steps):
    pred = np.zeros((steps, data.shape[1]))

    for b in tqdm(range(data.shape[1])):
        log_series = np.log1p(data[:, b])
        X = create_features_vectorized(log_series)
        y = log_series[168:]
        model = XGBRegressor(
            n_estimators=50,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            verbosity=0
        )
        model.fit(X, y)
        cur = log_series[-168:].copy()
        out = []
        for _ in range(steps):
            f = np.concatenate([
                cur,
                [cur[-24:].mean(), cur.mean(), cur[-24:].std()]
            ])
            p = model.predict(f.reshape(1,-1))[0]
            out.append(p)
            cur = np.roll(cur, -1)
            cur[-1] = p
        pred[:, b] = np.expm1(out)
    return pred

xgb_val = run_xgb(train_part, VAL_HORIZON)
xgb_test = run_xgb(train_data, len(test_data))

100%|██████████| 2880/2880 [08:30<00:00,  5.64it/s]


### Считаем метрики для xgboost на тестовой выборке

In [17]:
mae = mean_absolute_error(test_data.flatten(), xgb_test.flatten())
rmse = np.sqrt(np.mean((test_data - xgb_test)**2))

y_true = (test_data > THRESHOLD).flatten()
y_pred = (xgb_test > THRESHOLD).flatten()

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"XGBoost MAE: {mae:.4f}")
print(f"XGBoost RMSE: {rmse:.4f}")
print(f"XGBoost Precision: {precision:.4f}")
print(f"XGBoost Recall: {recall:.4f}")
print(f"XGBoost F1: {f1:.4f}")

XGBoost MAE: 0.2229
XGBoost RMSE: 0.3976
XGBoost Precision: 0.8585
XGBoost Recall: 0.8069
XGBoost F1: 0.8319


## Пробуем ансамбль моделей

In [18]:
def ensemble_loss(w):
    w = np.maximum(w, 0)
    w = w / w.sum()
    pred = w[0]*seasonal_val + w[1]*lr_val + w[2]*xgb_val
    return mean_absolute_error(val_part.flatten(), pred.flatten())

res = minimize(ensemble_loss, [0.3,0.3,0.4], bounds=[(0,1)]*3)
w = res.x / res.x.sum()

print("Weights:", w)

Weights: [0.1544187  0.44353809 0.40204321]


Интересно, самый большой вклад вносит линейная регрессия, а сезонная модель всего 0.15

In [19]:
ensemble_pred = (
        w[0]*seasonal_test +
        w[1]*lr_test +
        w[2]*xgb_test
)

### Считаем метрки ансамбля на тесте

In [20]:
mae = mean_absolute_error(test_data.flatten(), ensemble_pred.flatten())
rmse = np.sqrt(np.mean((test_data - ensemble_pred)**2))

y_true = (test_data > THRESHOLD).flatten()
y_pred = (ensemble_pred > THRESHOLD).flatten()

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Ensemble MAE: {mae:.4f}")
print(f"Ensemble RMSE: {rmse:.4f}")
print(f"Ensemble Precision: {precision:.4f}")
print(f"Ensemble Recall: {recall:.4f}")
print(f"Ensemble F1: {f1:.4f}")

Ensemble MAE: 0.2185
Ensemble RMSE: 0.3829
Ensemble Precision: 0.8475
Ensemble Recall: 0.8284
Ensemble F1: 0.8378


## Попробуем еще улучшить путем stacking

In [21]:
def time_series_folds(data, n_splits=4, val_size=168):
    folds = []
    total_len = len(data)
    for i in range(n_splits):
        train_end = total_len - val_size * (n_splits - i)
        val_start = train_end
        val_end = val_start + val_size
        if train_end <= 200:  # защита от слишком маленького трейна
            continue
        folds.append((0, train_end, val_start, val_end))
    return folds

In [22]:
n_models = 3  # seasonal, lr, xgb
oof_preds = np.zeros((len(train_data), train_data.shape[1], n_models))
folds = time_series_folds(train_data)
for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(folds):
    print(f"Fold {fold_id}")
    tr = train_data[tr_start:tr_end]
    val = train_data[val_start:val_end]
    # SEASONAL
    s = LearnableSeasonal()
    s.fit(tr, val)
    s_pred = s.predict(tr, len(val))
    # LR
    X_lr, y_lr = prepare_lr_features(tr)
    lr = Ridge(alpha=0.1)
    lr.fit(X_lr, y_lr)
    lr_pred_fold = rollout_lr(lr, tr[-24:], len(val))
    # XGB
    xgb_pred_fold = run_xgb(tr, len(val))
    oof_preds[val_start:val_end, :, 0] = s_pred
    oof_preds[val_start:val_end, :, 1] = lr_pred_fold
    oof_preds[val_start:val_end, :, 2] = xgb_pred_fold

Fold 0
Seasonal weights: 0.469, 0.496


100%|██████████| 2880/2880 [05:32<00:00,  8.65it/s]


Fold 1
Seasonal weights: 0.433, 0.492


100%|██████████| 2880/2880 [07:51<00:00,  6.11it/s]


Fold 2
Seasonal weights: 0.444, 0.506


100%|██████████| 2880/2880 [08:19<00:00,  5.76it/s]


Берем только те места, где есть OOF (последние куски)

In [23]:
valid_mask = oof_preds.sum(axis=2).sum(axis=1) != 0

X_meta = oof_preds[valid_mask].reshape(-1, n_models)
y_meta = train_data[valid_mask].reshape(-1)

In [24]:
meta = Ridge(alpha=1.0)
meta.fit(X_meta, y_meta)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [25]:
stack_input = np.stack([
    seasonal_test,
    lr_test,
    xgb_test
], axis=2)  # (time, beams, models)

X_test_meta = stack_input.reshape(-1, n_models)
meta_pred = meta.predict(X_test_meta)

# т.к. Ridge то reshape обратно
meta_pred = meta_pred.reshape(test_data.shape)

In [26]:
mae = mean_absolute_error(test_data.flatten(), meta_pred.flatten())
rmse = np.sqrt(np.mean((test_data - meta_pred)**2))

y_true = (test_data > THRESHOLD).flatten()
y_pred = (meta_pred > THRESHOLD).flatten()

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Stack MAE: {mae:.4f}")
print(f"Stack RMSE: {rmse:.4f}")
print(f"Stack Precision: {precision:.4f}")
print(f"Stack Recall: {recall:.4f}")
print(f"Stack F1: {f1:.4f}")

Stack MAE: 0.2216
Stack RMSE: 0.3823
Stack Precision: 0.8330
Stack Recall: 0.8433
Stack F1: 0.8381


# Выводы

## Сезонная модель ожидаемо сильна, но не оптимальна

Обучаемая сезонная модель (комбинация суточной и недельной сезонности) показала MAE = 0.2405 и F1 = 0.8161.
Это подтверждает наличие ярко выраженных сезонных паттернов в сетевом трафике (суточные и недельные циклы).
Однако веса сезонности (0.444 и 0.506) близки, что говорит о сопоставимом вкладе обоих горизонтов, и простая линейная комбинация уступает более сложным моделям.

## Линейная регрессия с rolling-окном показывает стабильные результаты

Ridge-регрессия с признаками (лаги 24 часа, средние и стандартные отклонения):

Модель оказалась лучше сезонной по F1 и recall, что говорит о способности улавливать не только сезонность, но и краткосрочные зависимости.
Интересно, что в ансамбле веса линейной регрессии оказались наибольшими (~0.44), что ***подчеркивает её важность как базового*** алгоритма для данной задачи.

## XGBoost — лучшая единичная модель

XGBoost превзошёл обе предыдущие модели:

Модель эффективно использует длинный горизонт (168 лагов) и агрегированные статистики.
Основной недостаток — вычислительная сложность.

## Взвешенный ансамбль даёт дополнительный прирост

Линейная комбинация трёх моделей с весами [0.154, 0.444, 0.402]:

MAE = 0.2185 (улучшение на ~2% относительно XGBoost)
F1 = 0.8378 (лучший показатель среди всех экспериментов)
Ансамбль компенсирует недостатки каждой модели: сезонная добавляет устойчивость к выбросам, линейная — интерпретируемость трендов, XGBoost — нелинейные зависимости.

## Стекинг (мета-модель на OOF-прогнозах) не превзошёл простой ансамбль

***Лучшее качество на тесте достигнуто взвешенным ансамблем, который сочетает интерпретируемость, учёт сезонности и нелинейных зависимостей.***